# Per què 2,45? La matemàtica dels arbres de decisió

**Optativa d'Aprenentatge automàtic · bloc de teoria**

A la demostració de la sessió 2 vau veure un arbre dibuixat amb un node a dalt de tot
que preguntava `petal_llarg <= 2.45`. La màquina va trobar aquest número sola,
mirant 105 flors d'entrenament.

La pregunta d'aquest quadern és directa: **per què 2,45, i per què el pètal i no una
altra columna?** Al final sabreu calcular-ho vosaltres mateixos, a mà i amb codi.

## 1. Les dades (les mateixes de sempre)

Tornem a carregar Iris, exactament com a la demo.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris

iris = load_iris(as_frame=True)
dades = iris.frame.copy()
dades["especie"] = iris.target_names[iris.target]
dades = dades.rename(columns={
    "sepal length (cm)": "sepal_llarg",
    "sepal width (cm)": "sepal_ample",
    "petal length (cm)": "petal_llarg",
    "petal width (cm)": "petal_ample",
})

X = dades[["sepal_llarg", "sepal_ample", "petal_llarg", "petal_ample"]]
y = dades["especie"]

print("Files:", len(dades))
dades.head()

## 2. Mesurar el desordre: la impuresa de Gini

Perquè una màquina triï un tall, primer li cal una manera de **mesurar si un grup de
flors està barrejat o no**. A això se li diu *impuresa*.

La idea, en paraules: un grup és **pur** si totes les flors són de la mateixa espècie.
És **impur** si hi ha una barreja, i com més equilibrada la barreja, més impur.

La fórmula que fa servir un arbre de decisió per defecte és la **impuresa de Gini**:

$$G = 1 - \sum_{i} p_i^2$$

on $p_i$ és la proporció de flors de la classe $i$ dins del grup. Si un grup té 30
setosa i 0 de la resta, $p_{setosa}=1$ i $G = 1 - 1^2 = 0$: totalment pur. Com més
barrejat el grup, més s'allunya $G$ de 0.

Provem-ho amb tres casos concrets.

In [ ]:
def gini(etiquetes):
    """Impuresa de Gini d'una llista/Series d'etiquetes."""
    etiquetes = pd.Series(etiquetes)
    if len(etiquetes) == 0:
        return 0.0
    proporcions = etiquetes.value_counts(normalize=True)
    return 1 - (proporcions ** 2).sum()


# Cas 1: grup totalment pur (50 setosa, cap més)
grup_pur = ["setosa"] * 50
print("50 setosa, 0 la resta  -> Gini =", gini(grup_pur))

# Cas 2: grup dividit a mitges entre dues classes
grup_meitats = ["setosa"] * 25 + ["versicolor"] * 25
print("25 setosa, 25 versicolor -> Gini =", gini(grup_meitats))

# Cas 3: tres classes a parts iguals
grup_tres = ["setosa"] * 10 + ["versicolor"] * 10 + ["virginica"] * 10
print("10+10+10, tres classes -> Gini =", round(gini(grup_tres), 3))

Els números surten exactament com s'esperava: **0** quan el grup és pur, **0,5** quan
és un empat a dues classes, i **≈0,667** quan hi ha tres classes igual de presents.
Com més classes barrejades i més equilibrades, més s'apropa Gini a 1.

### L'alternativa: l'entropia

Gini no és l'única manera de mesurar el desordre. L'altra habitual és l'**entropia**,
que ve de la teoria de la informació:

$$H = -\sum_i p_i \log_2(p_i)$$

Es calcula diferent (hi ha un logaritme de per mig) però mesura el mateix: 0 quan el
grup és pur, i un valor més alt com més barrejat està. A la pràctica, entrenar un
arbre amb Gini o amb entropia gairebé sempre dona **arbres pràcticament idèntics**;
per això scikit-learn fa servir Gini per defecte (és una mica més ràpid de calcular,
perquè no cal el logaritme). A l'exercici 1 la implementareu i ho comprovareu.

## 3. Com tria la màquina el tall

Ja sabem mesurar si *un* grup és pur. Ara ve la pregunta de debò: **d'entre totes les
columnes i tots els talls possibles, quin tria l'arbre?**

La idea és una cerca bruta. Per a cada columna, i per a cada possible llindar de tall,
es parteixen les 150 flors en dos grups (les que compleixen la condició i les que no)
i es calcula la impuresa **ponderada** dels dos fills:

$$G_{fills} = \frac{n_{esq}}{n} \, G_{esq} + \frac{n_{dre}}{n} \, G_{dre}$$

on $n_{esq}$ i $n_{dre}$ són quantes flors cauen a cada costat i $n$ el total. Es
pondera perquè no és el mateix deixar un grup pur de 2 flors que un de 80: el segon
val molt més.

El **guany** d'un tall és quant baixa la impuresa respecte al grup pare, abans de
partir-lo:

$$\text{guany} = G_{pare} - G_{fills}$$

L'arbre prova totes les combinacions (columna, llindar) i es queda amb la que dona
més guany. Implementem-ho.

In [ ]:
def gini_ponderat(y_esq, y_dre):
    """Impuresa de Gini dels dos fills, ponderada per la seva mida."""
    n = len(y_esq) + len(y_dre)
    return (len(y_esq) / n) * gini(y_esq) + (len(y_dre) / n) * gini(y_dre)


def millor_tall(X, y):
    """Prova totes les columnes i tots els llindars possibles i retorna el millor.

    Els llindars candidats són els punts mitjos entre valors consecutius que
    apareixen de debò a les dades: no té sentit provar-ne d'altres.
    """
    X = X.reset_index(drop=True)
    y = pd.Series(y).reset_index(drop=True)
    gini_pare = gini(y)

    millor = None
    for columna in X.columns:
        valors = np.sort(X[columna].unique())
        llindars = (valors[:-1] + valors[1:]) / 2
        for llindar in llindars:
            esq = y[X[columna] <= llindar]
            dre = y[X[columna] > llindar]
            if len(esq) == 0 or len(dre) == 0:
                continue
            guany = gini_pare - gini_ponderat(esq, dre)
            if millor is None or guany > millor["guany"]:
                millor = {"columna": columna, "llindar": llindar, "guany": guany}
    return millor


resultat = millor_tall(X, y)
print(f"Millor columna: {resultat['columna']}")
print(f"Millor llindar: {resultat['llindar']:.2f}")
print(f"Guany de Gini:  {resultat['guany']:.3f}")

**Aquí hi ha la resposta a la pregunta del principi.** La funció ha provat totes les
columnes i tots els talls possibles sobre les 150 flors, sense que ningú li digués res
d'iris, i ha trobat `petal_llarg` amb un llindar de **2,45**.

És exactament el node de dalt de l'arbre que vau veure a la demostració. No és
casualitat: és el mateix càlcul que fa `DecisionTreeClassifier` per dins, per triar el
primer tall.

## 4. Un gràfic del guany

Per entendre per què surt just 2,45 i no un altre número, mirem el guany de Gini que
donaria **cada** llindar possible sobre la columna del pètal, un per un.

In [ ]:
gini_pare = gini(y)
columna = "petal_llarg"

valors = np.sort(X[columna].unique())
llindars = (valors[:-1] + valors[1:]) / 2

guanys = []
for llindar in llindars:
    esq = y[X[columna] <= llindar]
    dre = y[X[columna] > llindar]
    guanys.append(gini_pare - gini_ponderat(esq, dre))

plt.figure(figsize=(8, 5))
plt.plot(llindars, guanys, color="tab:blue")
plt.axvline(resultat["llindar"], color="tab:red", linestyle="--",
            label=f"millor llindar = {resultat['llindar']:.2f}")
plt.xlabel("Llindar de tall sobre petal_llarg (cm)")
plt.ylabel("Guany de Gini")
plt.title("Guany de cada llindar possible, columna petal_llarg")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

El guany puja de cop cap a 1,5-2 cm, es queda dalt de tot en un replà una mica ample
(entre les setosa, que tenen el pètal curt, i la resta, hi ha un buit sencer sense cap
flor) i torna a baixar després. El llindar de 2,45 cau just al mig d'aquest replà: és
el punt exacte on hi ha el buit més gran entre la darrera setosa i la primera flor
d'una altra espècie. Per això la línia vermella queda dins de la zona plana i no en un
pic estret: qualsevol llindar d'aquest replà separaria les setosa igual de bé, i
l'algorisme es queda amb el del mig.

## 5. Comparem amb scikit-learn

Si tot això és el que fa un arbre de decisió per dins, entrenar-ne un de debò amb
`max_depth=1` (un únic tall) hauria de triar exactament el mateix llindar.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

arbre = DecisionTreeClassifier(max_depth=1, random_state=42)
arbre.fit(X, y)

columna_sklearn = X.columns[arbre.tree_.feature[0]]
llindar_sklearn = arbre.tree_.threshold[0]

print(f"scikit-learn tria: {columna_sklearn} <= {llindar_sklearn:.2f}")
print(f"la nostra funció:  {resultat['columna']} <= {resultat['llindar']:.2f}")

Coincideixen. `arbre.tree_.feature` i `arbre.tree_.threshold` guarden, per a cada
node de l'arbre, quina columna i quin llindar ha triat scikit-learn; el node 0 és
sempre l'arrel. `millor_tall` no és una aproximació ni una simplificació docent: és
**el mateix algorisme**, escrit a mà.

## 6. Fins on fer créixer l'arbre

Un sol tall (`max_depth=1`) no separa bé les tres espècies: només distingeix setosa de
la resta. La solució òbvia és repetir el mateix procés dins de cada fill, una vegada i
una altra: és així com creix un arbre, tall rere tall.

La pregunta és: **fins on?** Si no s'atura mai, l'arbre acaba fent un tall per a cada
flor individual, memoritza les dades d'entrenament i deixa de generalitzar. Mirem-ho
amb dades de veritat: entrenem arbres de profunditat 1 a 10 i comparem la precisió en
entrenament amb la de l'examen.

In [ ]:
from sklearn.model_selection import train_test_split

X_entrena, X_examen, y_entrena, y_examen = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

profunditats = range(1, 11)
precisio_entrena = []
precisio_examen = []

for p in profunditats:
    a = DecisionTreeClassifier(max_depth=p, random_state=42)
    a.fit(X_entrena, y_entrena)
    precisio_entrena.append(a.score(X_entrena, y_entrena))
    precisio_examen.append(a.score(X_examen, y_examen))

plt.figure(figsize=(8, 5))
plt.plot(profunditats, precisio_entrena, marker="o", label="Entrenament")
plt.plot(profunditats, precisio_examen, marker="o", label="Examen")
plt.xlabel("max_depth")
plt.ylabel("Precisió")
plt.title("Precisió segons la profunditat de l'arbre")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

pd.DataFrame({"max_depth": list(profunditats),
              "entrenament": [f"{v:.1%}" for v in precisio_entrena],
              "examen": [f"{v:.1%}" for v in precisio_examen]})

Amb `max_depth=1` l'arbre encerta un 66,7 % en tots dos costats: només separa setosa,
i les altres dues espècies queden barrejades. A `max_depth=3` arriba al **97,8 %**
d'examen, el millor de tots. A partir d'aquí la precisió d'**entrenament** segueix
pujant fins arribar al **100 %** (l'arbre ja té prou talls per aïllar cada flor
d'entrenament una per una), mentre que la d'**examen** es queda enrere, al voltant del
93 %, sense tornar a millorar.

Això és el **sobreajust** (*overfitting*): l'arbre deixa d'aprendre el patró general
de les espècies i comença a memoritzar particularitats de les 105 flors concretes que
ha vist entrenant. Un arbre profund és perfecte amb els exemples que ja coneix i pitjor
amb els que no.

Per evitar-ho, un arbre de decisió es **poda**, limitant fins on pot créixer:

- **`max_depth`**: quants talls seguits pot fer com a màxim des de l'arrel.
- **`min_samples_leaf`**: quantes flors com a mínim ha de tenir cada fulla final. Si un
  tall deixaria una fulla amb menys flors que aquest número, no es fa. Evita que
  l'arbre acabi creant una fulla només per a una o dues flors soltes.

Cap dels dos té un valor "correcte" universal: es prova amb l'examen, com acabem de
fer amb el gràfic de dalt.

## 7. Pràctica

Quatre exercicis. Als tres primers, treballeu sobre el que ja hi ha en aquest quadern
(`gini`, `gini_ponderat`, `millor_tall`, `X`, `y`, `dades`).

### Exercici 1 — Implementa l'entropia

Implementa `entropia(etiquetes)` seguint la fórmula $H = -\sum_i p_i \log_2(p_i)$
(`np.log2` us servirà). Comprova que dona 0 per a un grup pur i un valor màxim per a
un grup repartit a parts iguals, igual que vau veure amb Gini.

Després, escriviu una versió de `millor_tall` que faci servir l'entropia en lloc de
Gini (podeu copiar-la i canviar només la mesura d'impuresa) i comproveu si tria el
mateix tall que la versió amb Gini.

In [ ]:
def entropia(etiquetes):
    # TODO: implementeu la formula H = -sum(p_i * log2(p_i))
    # pista: value_counts(normalize=True) us dona les p_i
    pass


# Comproveu-la amb els mateixos tres grups de la secció 2
# print(entropia(["setosa"] * 50))
# print(entropia(["setosa"] * 25 + ["versicolor"] * 25))
# print(entropia(["setosa"] * 10 + ["versicolor"] * 10 + ["virginica"] * 10))

# TODO: adapteu millor_tall perque faci servir entropia() en lloc de gini(),
# executeu-la sobre X, y i compareu el resultat amb `resultat`

### Exercici 2 — Construïu a mà un arbre de profunditat 2

`millor_tall` només fa un tall. Un arbre de profunditat 2 és el mateix procés aplicat
dues vegades: primer sobre totes les dades, i després **per separat** dins de cada
grup que ha quedat del primer tall.

1. Apliqueu `millor_tall(X, y)` per obtenir el primer tall (ja el teniu a `resultat`).
2. Amb aquest tall, separeu `X` i `y` en dos grups (esquerra i dreta).
3. Apliqueu `millor_tall` **una altra vegada, per separat, a cada grup**.
4. Escriviu els tres talls que heu trobat com si fossin `if`/`elif`/`else`, i
   compareu-los amb l'arbre de `plot_tree` que vau veure a la demostració.

In [ ]:
# Pas 1: el primer tall ja el teniu a `resultat`

# Pas 2: separeu X i y en grup esquerra (compleix la condicio) i grup dreta
# grup_esq = X[columna] <= llindar   (feu servir resultat["columna"] i resultat["llindar"])
# grup_dre = ...

# Pas 3: crideu millor_tall dins de cada grup per separat

# Pas 4: escriviu els if/elif/else que en resulten, com al quadern de la demo

### Exercici 3 — Un altre dataset: Wine

`sklearn.datasets.load_wine()` té 178 mostres de vi, 13 columnes químiques i 3 classes
(tres cellers diferents). Carregueu-lo amb `load_wine(as_frame=True)`, monteu un `X` i
un `y` com hem fet amb Iris, i apliqueu-hi `millor_tall`. Quina columna i quin llindar
tria per al primer tall? Entrenat un `DecisionTreeClassifier(max_depth=1)` sobre les
mateixes dades i comproveu que coincideix.

In [ ]:
from sklearn.datasets import load_wine

# vi = load_wine(as_frame=True)
# X_vi = vi.frame[vi.feature_names]
# y_vi = vi.frame["target"]

# TODO: apliqueu millor_tall(X_vi, y_vi) i mireu quina columna i llindar surten

# TODO: entreneu un DecisionTreeClassifier(max_depth=1) sobre X_vi, y_vi
# i compareu arbre.tree_.feature / arbre.tree_.threshold amb el resultat de dalt

### Exercici 4 — Trobeu la profunditat òptima

A la secció 6 vau veure la precisió d'examen per a `max_depth` d'1 a 10, però no vau
buscar-la de manera automàtica. Feu-ho: a partir de les llistes `precisio_examen` i
`profunditats` (o tornant a calcular-les), trobeu **quina profunditat dona la millor
precisió d'examen** i imprimiu-la. Compareu-la amb el que veieu al gràfic.

In [ ]:
# TODO: a partir de precisio_examen i profunditats, trobeu l'index del maxim
# pista: np.argmax(precisio_examen) us dona la posicio, no el valor de max_depth

# TODO: imprimiu la profunditat optima i la seva precisio d'examen

## Resum

- Un arbre de decisió tria cada tall mesurant la **impuresa de Gini** abans i després
  de partir les dades, i quedant-se amb el llindar que dona **més guany**.
- Aquest càlcul, fet a mà amb `millor_tall`, troba exactament els mateixos talls que
  `DecisionTreeClassifier`: **`petal_llarg <= 2,45`** per al primer node, el mateix que
  vau veure dibuixat a la demostració.
- Un arbre sense límit de profunditat arriba al 100 % en entrenament però es queda
  enrere a l'examen: és **sobreajust**. `max_depth` i `min_samples_leaf` el controlen.

### El següent pas

Un sol arbre, per ben triat que tingui cada tall, és inestable: canvieu una mica les
dades d'entrenament i pot sortir un arbre diferent. La solució que ja vau veure a la
demostració (el `RandomForestClassifier`) és entrenar-ne **cent alhora**, cadascun amb
una part diferent de les dades, i fer-los votar. Això és el que ve al quadern
següent: **boscos aleatoris**.